# Fraud Detection Training

Train Isolation Forest on transaction-level features for anti-gaming detection.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest
import pickle
from pathlib import Path

# Import our modules
import sys
sys.path.append('../')
from training.synthetic_data_gen import generate_synthetic_data
from training.fraud_feature_engine import compute_transaction_features

## Generate Synthetic Transaction Data

In [ ]:
# Generate synthetic data for multiple users
np.random.seed(42)
n_users = 1000
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 1, 1)

all_txns = []
for user_id in range(n_users):
    user_txns = generate_synthetic_data(
        user_id=str(user_id),
        start_date=start_date,
        end_date=end_date,
        archetype=None  # random archetype
    )
    all_txns.append(user_txns)

transaction_df = pd.concat(all_txns, ignore_index=True)
print(f"Generated {len(transaction_df)} transactions for {n_users} users")
transaction_df.head()

## Compute Transaction-Level Features

In [ ]:
# Compute features for all transactions
# For training, we use the end_date as as_of
as_of = end_date

# Group by user and compute features
feature_dfs = []
for user_id, user_txns in transaction_df.groupby('user_id'):
    user_features = compute_transaction_features(user_txns, as_of)
    if len(user_features) > 0:
        feature_dfs.append(user_features)

all_features = pd.concat(feature_dfs, ignore_index=True)
print(f"Computed features for {len(all_features)} transactions")
all_features.head()

## Train Isolation Forest Model

In [ ]:
from inference.fraud_predictor import FraudPredictor

# Train the model
contamination = 0.05  # assume 5% fraud
fraud_predictor = FraudPredictor.train(all_features, contamination=contamination)

print("Isolation Forest trained successfully")

## Compute Anomaly Scores

In [ ]:
# Predict anomaly scores
features_with_scores = fraud_predictor.predict_transaction_anomalies(all_features)
print(f"Anomaly scores computed. Mean: {features_with_scores['anomaly_score'].mean():.3f}, Max: {features_with_scores['anomaly_score'].max():.3f}")
features_with_scores[['anomaly_score']].describe()

## Save the Trained Model

In [ ]:
# Load existing artifact and add fraud model
artifact_path = Path('../models/fraud_model_artifact_v1.pkl')
with open(artifact_path, 'rb') as f:
    artifact = pickle.load(f)

# Add fraud model
artifact['fraud_model'] = fraud_predictor.model

# Save updated artifact
with open(artifact_path, 'wb') as f:
    pickle.dump(artifact, f)

print("Fraud model added to artifact and saved")